# TP 06 — Seq2Seq with Bahdanau Attention

**Course:** Natural Language Processing  
**Session:** 6 / 8

---

## Context

In S05 you built LSTM classifiers. The encoder compressed its input into a single hidden state.
Today you build a **sequence-to-sequence** model with **Bahdanau attention**: the decoder can
look at every encoder state at each generation step, with learned weights.

**Task:** English → French translation on a small controlled corpus.

**Key result:** a visualisable attention map showing word-level alignment
and adjective reordering (EN: *the black cat* → FR: *le chat noir*).

---

## Roadmap

| Part | Task | New concept |
|------|------|-------------|
| 1 | Load corpus, build vocabularies | tokenisation, SOS/EOS tokens |
| 2 | Encode sentences as integer sequences | padding, integer encoding |
| 3 | Build the encoder (Functional API) | `return_sequences`, `return_state` |
| 4 | Implement `BahdanauAttention` | custom Keras `Layer`, alignment scores |
| 5 | Build and train the full seq2seq model | teacher forcing |
| 6 | Greedy decoding and attention map | inference loop, heatmap |
| 7 (bonus) | Beam search | width-k decoding |

---


## Setup


In [ ]:
# !pip install tensorflow matplotlib scikit-learn -q

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import Counter

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense, Concatenate, Layer
)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ── Hyperparameters ───────────────────────────────────────────────────────────
EMB_DIM    = 64
LSTM_UNITS = 128
EPOCHS     = 60
BATCH_SIZE = 32

print(f'TensorFlow: {tf.__version__}')
print('Setup OK.')


---
## Part 1 — Corpus and Vocabularies


In [ ]:
EN_FR_CORPUS = [
    ("the cat sits", "le chat est assis"),
    ("the black cat sits", "le chat noir est assis"),
    ("the white cat sleeps", "le chat blanc dort"),
    ("the small dog runs", "le petit chien court"),
    ("the big dog eats", "le grand chien mange"),
    ("the black dog barks", "le chien noir aboie"),
    ("the white dog sleeps", "le chien blanc dort"),
    ("the cat eats", "le chat mange"),
    ("the dog runs", "le chien court"),
    ("the cat runs", "le chat court"),
    ("the dog sleeps", "le chien dort"),
    ("the dog barks", "le chien aboie"),
    ("a cat sits", "un chat est assis"),
    ("a black cat runs", "un chat noir court"),
    ("a white dog sleeps", "un chien blanc dort"),
    ("a small cat eats", "un petit chat mange"),
    ("the small cat plays", "le petit chat joue"),
    ("the big cat sleeps", "le grand chat dort"),
    ("the red bird sings", "l oiseau rouge chante"),
    ("a red bird flies", "un oiseau rouge vole"),
    ("the blue bird flies", "l oiseau bleu vole"),
    ("a blue bird sings", "un oiseau bleu chante"),
    ("the bird sings", "l oiseau chante"),
    ("the bird flies", "l oiseau vole"),
    ("the fish swims", "le poisson nage"),
    ("the small fish swims", "le petit poisson nage"),
    ("one cat", "un chat"),
    ("two dogs", "deux chiens"),
    ("three birds", "trois oiseaux"),
    ("four fish", "quatre poissons"),
    ("five cats", "cinq chats"),
    ("one dog runs", "un chien court"),
    ("two cats sleep", "deux chats dorment"),
    ("three dogs run", "trois chiens courent"),
    ("two birds sing", "deux oiseaux chantent"),
    ("three fish swim", "trois poissons nagent"),
    ("the cat is in the garden", "le chat est dans le jardin"),
    ("the dog is in the garden", "le chien est dans le jardin"),
    ("the cat sits in the garden", "le chat est assis dans le jardin"),
    ("the dog runs in the garden", "le chien court dans le jardin"),
    ("the bird sings in the garden", "l oiseau chante dans le jardin"),
    ("the cat is on the mat", "le chat est sur le tapis"),
    ("the dog is on the mat", "le chien est sur le tapis"),
    ("the black cat is on the mat", "le chat noir est sur le tapis"),
    ("the white dog is on the mat", "le chien blanc est sur le tapis"),
    ("the cat sits on the mat", "le chat est assis sur le tapis"),
    ("the dog sleeps on the mat", "le chien dort sur le tapis"),
    ("the cat is in the house", "le chat est dans la maison"),
    ("the dog is in the house", "le chien est dans la maison"),
    ("the bird flies over the house", "l oiseau vole au dessus de la maison"),
    ("the cat sleeps in the house", "le chat dort dans la maison"),
    ("the black cat", "le chat noir"),
    ("the white cat", "le chat blanc"),
    ("the black dog", "le chien noir"),
    ("the white dog", "le chien blanc"),
    ("the red bird", "l oiseau rouge"),
    ("the blue bird", "l oiseau bleu"),
    ("the small cat", "le petit chat"),
    ("the big dog", "le grand chien"),
    ("the small dog", "le petit chien"),
    ("the big cat", "le grand chat"),
    ("the cat eats the fish", "le chat mange le poisson"),
    ("the dog eats the fish", "le chien mange le poisson"),
    ("the cat chases the bird", "le chat chasse l oiseau"),
    ("the dog chases the cat", "le chien chasse le chat"),
    ("the cat drinks the water", "le chat boit l eau"),
    ("the dog drinks the water", "le chien boit l eau"),
    ("the bird eats the fish", "l oiseau mange le poisson"),
    ("i see the cat", "je vois le chat"),
    ("i see the dog", "je vois le chien"),
    ("i see the bird", "je vois l oiseau"),
    ("i see the fish", "je vois le poisson"),
    ("i like the cat", "j aime le chat"),
    ("i like the dog", "j aime le chien"),
    ("i like the bird", "j aime l oiseau"),
    ("she sees the cat", "elle voit le chat"),
    ("she sees the dog", "elle voit le chien"),
    ("he sees the bird", "il voit l oiseau"),
    ("she likes the cat", "elle aime le chat"),
    ("he likes the dog", "il aime le chien"),
    ("i see the black cat", "je vois le chat noir"),
    ("she sees the white dog", "elle voit le chien blanc"),
    ("he sees the red bird", "il voit l oiseau rouge"),
    ("i like the small cat", "j aime le petit chat"),
    ("she likes the big dog", "elle aime le grand chien"),
    ("the cat sleeps today", "le chat dort aujourd hui"),
    ("the dog runs today", "le chien court aujourd hui"),
    ("the bird sings today", "l oiseau chante aujourd hui"),
    ("the cat eats today", "le chat mange aujourd hui"),
    ("the dog barks today", "le chien aboie aujourd hui"),
    ("the black cat sits in the garden", "le chat noir est assis dans le jardin"),
    ("the white dog runs in the garden", "le chien blanc court dans le jardin"),
    ("the red bird flies in the garden", "l oiseau rouge vole dans le jardin"),
    ("the black cat is on the mat", "le chat noir est sur le tapis"),
    ("the white dog sleeps in the house", "le chien blanc dort dans la maison"),
    ("the small cat is in the garden", "le petit chat est dans le jardin"),
    ("the big dog runs in the garden", "le grand chien court dans le jardin"),
    ("one black cat", "un chat noir"),
    ("two white dogs", "deux chiens blancs"),
    ("three red birds", "trois oiseaux rouges"),
    ("one small cat runs", "un petit chat court"),
    ("two big dogs sleep", "deux grands chiens dorment"),
    ("three black cats sit", "trois chats noirs sont assis"),
    ("the black cat eats the fish", "le chat noir mange le poisson"),
    ("the white dog chases the cat", "le chien blanc chasse le chat"),
    ("the small cat drinks the water", "le petit chat boit l eau"),
    ("the big dog eats the fish", "le grand chien mange le poisson"),
    ("the red bird sings in the garden", "l oiseau rouge chante dans le jardin"),
    ("the blue bird flies over the house", "l oiseau bleu vole au dessus de la maison"),
    ("i see the black cat in the garden", "je vois le chat noir dans le jardin"),
    ("she sees the white dog on the mat", "elle voit le chien blanc sur le tapis"),
    ("he sees the red bird in the garden", "il voit l oiseau rouge dans le jardin"),
    ("the cat does not eat", "le chat ne mange pas"),
    ("the dog does not bark", "le chien n aboie pas"),
    ("the bird does not sing", "l oiseau ne chante pas"),
    ("the cat does not sleep", "le chat ne dort pas"),
    ("the dog does not run", "le chien ne court pas"),
    ("the cat does not like the dog", "le chat n aime pas le chien"),
    ("the dog does not like the cat", "le chien n aime pas le chat"),
    ("where is the cat", "ou est le chat"),
    ("where is the dog", "ou est le chien"),
    ("where is the bird", "ou est l oiseau"),
    ("what does the cat eat", "que mange le chat"),
    ("what does the dog eat", "que mange le chien"),
    ("the black bird sings", "l oiseau noir chante"),
    ("the white bird flies", "l oiseau blanc vole"),
    ("a black cat eats", "un chat noir mange"),
    ("a white dog barks", "un chien blanc aboie"),
    ("the small bird sings", "le petit oiseau chante"),
    ("the big fish swims", "le grand poisson nage"),
    ("a small bird flies", "un petit oiseau vole"),
    ("a big fish swims", "un grand poisson nage"),
    ("the red cat sits", "le chat rouge est assis"),
    ("the blue cat sleeps", "le chat bleu dort"),
    ("two black cats", "deux chats noirs"),
    ("three white dogs", "trois chiens blancs"),
    ("one blue bird sings", "un oiseau bleu chante"),
    ("two small cats sleep", "deux petits chats dorment"),
    ("three big dogs run", "trois grands chiens courent"),
    ("the black cat sits on the mat", "le chat noir est assis sur le tapis"),
    ("the white cat sits on the mat", "le chat blanc est assis sur le tapis"),
    ("the small dog sleeps on the mat", "le petit chien dort sur le tapis"),
    ("the red bird sings in the house", "l oiseau rouge chante dans la maison"),
    ("the blue bird flies over the garden", "l oiseau bleu vole au dessus du jardin"),
    ("the fish swims in the water", "le poisson nage dans l eau"),
    ("the small fish swims in the water", "le petit poisson nage dans l eau"),
    ("the big fish eats", "le grand poisson mange"),
    ("i see two cats", "je vois deux chats"),
    ("she sees three dogs", "elle voit trois chiens"),
    ("he sees one bird", "il voit un oiseau"),
    ("we see the cat", "nous voyons le chat"),
    ("we like the dog", "nous aimons le chien"),
    ("they see the bird", "ils voient l oiseau"),
    ("they like the fish", "ils aiment le poisson"),
    ("the cat is small", "le chat est petit"),
    ("the dog is big", "le chien est grand"),
    ("the bird is red", "l oiseau est rouge"),
    ("the fish is small", "le poisson est petit"),
    ("the cat is black", "le chat est noir"),
    ("the dog is white", "le chien est blanc"),
    ("the bird is blue", "l oiseau est bleu"),
    ("the cat is happy", "le chat est heureux"),
    ("the dog is happy", "le chien est heureux"),
    ("the bird is beautiful", "l oiseau est beau"),
]

print(f'Total pairs: {len(EN_FR_CORPUS)}')
print(f'Example pairs:')
for en, fr in EN_FR_CORPUS[:5]:
    print(f'  EN: {en}')
    print(f'  FR: {fr}')
    print()


### 1.1 — Implement `build_vocab`

Build a word-to-index mapping from a list of sentences.
Reserve indices 0–3 for special tokens:

| Index | Token | Meaning |
|-------|-------|---------|
| 0 | `<PAD>` | Padding |
| 1 | `<SOS>` | Start of sequence |
| 2 | `<EOS>` | End of sequence |
| 3 | `<UNK>` | Unknown word |


In [ ]:
def build_vocab(
    sentences: list[str],
) -> tuple[dict, dict]:
    """
    Build word-to-index and index-to-word mappings.

    Special tokens occupy indices 0–3:
        0: <PAD>, 1: <SOS>, 2: <EOS>, 3: <UNK>
    Remaining words are assigned indices starting from 4,
    ordered by frequency (most frequent first).

    Parameters
    ----------
    sentences : list of str
        Tokenised sentences (lowercase, whitespace-split).

    Returns
    -------
    word2idx : dict
    idx2word : dict
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
src_sentences = [en for en, fr in EN_FR_CORPUS]
tgt_sentences = [fr for en, fr in EN_FR_CORPUS]

src_w2i, src_i2w = build_vocab(src_sentences)
tgt_w2i, tgt_i2w = build_vocab(tgt_sentences)

SRC_VOCAB = len(src_w2i)
TGT_VOCAB = len(tgt_w2i)

PAD, SOS, EOS, UNK = 0, 1, 2, 3

print(f'Source vocab size: {SRC_VOCAB}')
print(f'Target vocab size: {TGT_VOCAB}')
print(f'Sample src entries: {list(src_w2i.items())[:6]}')
print(f'Sample tgt entries: {list(tgt_w2i.items())[:6]}')


---
## Part 2 — Encode Sentences


### 2.1 — Implement `encode_sentences`

Convert a list of sentences to a padded integer matrix.
Each sentence should be wrapped with `<SOS>` and `<EOS>` tokens.


In [ ]:
def encode_sentences(
    sentences: list[str],
    word2idx: dict,
    max_len: int | None = None,
) -> np.ndarray:
    """
    Convert sentences to a padded integer matrix.

    Each sentence is wrapped with SOS (index 1) and EOS (index 2).
    Unknown words map to UNK (index 3).
    Sequences are post-padded with PAD (index 0) to max_len.
    If max_len is None, use the length of the longest sentence + 2.

    Parameters
    ----------
    sentences : list of str
    word2idx : dict
    max_len : int or None

    Returns
    -------
    np.ndarray, shape (len(sentences), max_len)
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Determine max lengths
src_max_len = max(len(s.split()) for s in src_sentences) + 2
tgt_max_len = max(len(t.split()) for t in tgt_sentences) + 2

# Encode
src_enc = encode_sentences(src_sentences, src_w2i, src_max_len)
tgt_enc = encode_sentences(tgt_sentences, tgt_w2i, tgt_max_len)

# Train / val split
(
    src_train, src_val,
    tgt_train, tgt_val,
) = train_test_split(
    src_enc, tgt_enc,
    test_size=0.15, random_state=RANDOM_STATE
)

print(f'src_train: {src_train.shape}   src_val: {src_val.shape}')
print(f'tgt_train: {tgt_train.shape}   tgt_val: {tgt_val.shape}')
print()
print(f'Example encoded source: {src_enc[0]}')
print(f'Decoded: {" ".join(src_i2w.get(i, "?") for i in src_enc[0] if i != 0)}')


---
## Part 3 — Encoder (Keras Functional API)

The encoder must return **all** hidden states (for attention) **and** the final state
(to initialise the decoder). `Sequential` cannot express this — use `Model(inputs, outputs)`.


In [ ]:
def build_encoder(
    src_vocab: int,
    emb_dim: int,
    lstm_units: int,
    src_max_len: int,
) -> Model:
    """
    Build the encoder as a Keras functional Model.

    Architecture
    ------------
    Input(src_max_len,)
    Embedding(src_vocab, emb_dim, mask_zero=True)
    LSTM(lstm_units, return_sequences=True, return_state=True)

    The model returns three outputs:
        enc_out : (batch, src_max_len, lstm_units)  — all hidden states
        enc_h   : (batch, lstm_units)               — final hidden state
        enc_c   : (batch, lstm_units)               — final cell state

    Parameters
    ----------
    src_vocab : int
    emb_dim : int
    lstm_units : int
    src_max_len : int

    Returns
    -------
    tf.keras.Model with outputs [enc_out, enc_h, enc_c]
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
encoder = build_encoder(SRC_VOCAB, EMB_DIM, LSTM_UNITS, src_max_len)
encoder.summary()
print()
# Quick shape test
dummy = np.zeros((2, src_max_len), dtype=np.int32)
out, h, c = encoder(dummy)
print(f'enc_out shape: {out.shape}  (batch, src_len, units)')
print(f'enc_h   shape: {h.shape}   (batch, units)')
print(f'enc_c   shape: {c.shape}   (batch, units)')


---
## Part 4 — BahdanauAttention Layer

Implement the attention mechanism as a custom Keras `Layer` subclass.

At each decoder step, given:
- `enc_out`: all encoder hidden states, shape `(batch, src_len, units)`
- `dec_hidden`: current decoder hidden state, shape `(batch, units)`

The layer computes:

```
e_{i}  = vᵀ · tanh( W1 · enc_out_i + W2 · dec_hidden )    # alignment score
alpha  = softmax( e )                                        # attention weights
context = Σ alpha_i · enc_out_i                             # context vector
```


In [ ]:
class BahdanauAttention(Layer):
    """
    Bahdanau (additive) attention layer.

    Parameters
    ----------
    units : int
        Dimension of the alignment hidden layer (typically = lstm_units).

    Call inputs
    -----------
    enc_out    : tf.Tensor, shape (batch, src_len, lstm_units)
        All encoder hidden states.
    dec_hidden : tf.Tensor, shape (batch, lstm_units)
        Current decoder hidden state.

    Returns
    -------
    context : tf.Tensor, shape (batch, lstm_units)
        Weighted sum of encoder states.
    alpha   : tf.Tensor, shape (batch, src_len, 1)
        Attention weights (softmax over source positions).

    Notes
    -----
    Trainable weights: W1 (Dense, units), W2 (Dense, units), V (Dense, 1).
    W1 projects enc_out, W2 projects dec_hidden (expanded to match src_len),
    V projects the tanh combination to a scalar score per position.
    """

    def __init__(self, units: int, **kwargs):
        super().__init__(**kwargs)
        # YOUR CODE HERE: define self.W1, self.W2, self.V as Dense layers
        raise NotImplementedError

    def call(
        self,
        enc_out: tf.Tensor,
        dec_hidden: tf.Tensor,
    ) -> tuple[tf.Tensor, tf.Tensor]:
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
# Quick shape test
attn = BahdanauAttention(LSTM_UNITS)
dummy_enc = tf.zeros((2, src_max_len, LSTM_UNITS))
dummy_dec = tf.zeros((2, LSTM_UNITS))
ctx, alp = attn(dummy_enc, dummy_dec)
print(f'context shape : {ctx.shape}   (batch, lstm_units)')
print(f'alpha shape   : {alp.shape}  (batch, src_len, 1)')
print(f'alpha sum     : {tf.reduce_sum(alp, axis=1).numpy().flatten()[:2]}  (should be ~1.0)')


---
## Part 5 — Seq2Seq Model with Teacher Forcing

**Teacher forcing:** during training, the decoder receives the ground-truth target token
at each step (not its own previous prediction). This gives a clean gradient signal.

At each decoder step `t`:
1. Compute context vector `cₜ` via `BahdanauAttention(enc_out, dec_h)`
2. Concatenate `[target_embed_t ; cₜ]` as input to the decoder LSTM
3. Pass through a Dense(TGT_VOCAB, softmax) to get logits

The loop runs over all target positions simultaneously (vectorised with tf.TensorArray).


In [ ]:
def build_seq2seq_model(
    encoder: Model,
    tgt_vocab: int,
    emb_dim: int,
    lstm_units: int,
    tgt_max_len: int,
) -> tuple[Model, 'BahdanauAttention', object, object]:
    """
    Build the full seq2seq model with Bahdanau attention and teacher forcing.

    Parameters
    ----------
    encoder : tf.keras.Model
        Pre-built encoder (from build_encoder).
    tgt_vocab : int
    emb_dim : int
    lstm_units : int
    tgt_max_len : int

    Returns
    -------
    model         : tf.keras.Model — full training model
    attn_layer    : BahdanauAttention — reused at inference
    dec_embedding : Embedding layer  — reused at inference
    dec_lstm      : LSTM layer        — reused at inference
    dec_dense     : Dense layer       — reused at inference

    Notes
    -----
    Inputs: [src_input (batch, src_len), tgt_input (batch, tgt_len)]
    Output: (batch, tgt_len, tgt_vocab) — log probabilities at each step.
    Training loss: sparse categorical cross-entropy ignoring PAD tokens.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
model, attn_layer, dec_emb, dec_lstm, dec_dense = build_seq2seq_model(
    encoder, TGT_VOCAB, EMB_DIM, LSTM_UNITS, tgt_max_len
)
model.summary()


In [ ]:
# Decoder input: target shifted right (SOS + tokens), decoder output: tokens + EOS
# tgt_train[:,:-1] = decoder input (starts with SOS)
# tgt_train[:,1:]  = decoder target (ends with EOS)

history = model.fit(
    [src_train, tgt_train[:, :-1]],
    tgt_train[:, 1:],
    validation_data=(
        [src_val, tgt_val[:, :-1]],
        tgt_val[:, 1:],
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0,
)

# Plot training curve
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, metric in zip(axes, ['loss', 'accuracy']):
    ax.plot(history.history[metric],     label='train', color='#56c8ba')
    ax.plot(history.history[f'val_{metric}'], label='val',   color='#e8c47a')
    ax.set_title(metric, color='#e6edf3')
    ax.legend()
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    ax.spines[:].set_color('#30363d')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()


---
## Part 6 — Greedy Decoding and Attention Map

At inference there is no ground-truth to feed the decoder.
Use **greedy decoding**: at each step, take the argmax of the output distribution
and feed it as input to the next step. Stop when `<EOS>` is generated or `tgt_max_len` is reached.

Collect the attention weights `alpha` at each step to build the attention map.


In [ ]:
def translate(
    sentence: str,
    encoder: Model,
    attn_layer: 'BahdanauAttention',
    dec_emb: object,
    dec_lstm: object,
    dec_dense: object,
    src_w2i: dict,
    tgt_i2w: dict,
    src_max_len: int,
    tgt_max_len: int,
) -> tuple[str, np.ndarray]:
    """
    Translate a single English sentence using greedy decoding.

    Steps:
        1. Encode the source sentence with the encoder.
        2. Initialise the decoder hidden state with the encoder final state.
        3. Start with SOS token.
        4. At each step: compute attention, run decoder LSTM, argmax output.
        5. Stop at EOS or max length.

    Parameters
    ----------
    sentence : str
        Raw English sentence (lowercase, whitespace-tokenised).
    encoder, attn_layer, dec_emb, dec_lstm, dec_dense : model components
    src_w2i, tgt_i2w : vocabulary dicts
    src_max_len, tgt_max_len : int

    Returns
    -------
    translation : str
    attention   : np.ndarray, shape (tgt_len, src_len)
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
test_sentences = [
    'the black cat sits on the mat',
    'the white dog runs in the garden',
    'the red bird sings',
    'the cat does not eat',
    'two black cats',
]

for sent in test_sentences:
    translation, _ = translate(
        sent, encoder, attn_layer, dec_emb, dec_lstm, dec_dense,
        src_w2i, tgt_i2w, src_max_len, tgt_max_len,
    )
    print(f'EN: {sent}')
    print(f'FR: {translation}')
    print()


### 6.1 — Implement `plot_attention_map`


In [ ]:
def plot_attention_map(
    sentence: str,
    translation: str,
    attention: np.ndarray,
    figsize: tuple = (8, 6),
) -> plt.Figure:
    """
    Plot an attention map as a heatmap.

    Rows = target tokens, Columns = source tokens.
    Cell (j, i) = attention weight alpha_{t=j, i}.

    Parameters
    ----------
    sentence : str
        Source sentence (space-separated tokens).
    translation : str
        Generated translation (space-separated tokens).
    attention : np.ndarray, shape (tgt_len, src_len)
    figsize : tuple

    Returns
    -------
    plt.Figure

    Notes
    -----
    Use plt.imshow with cmap='YlOrRd' or similar.
    Label x-axis with source tokens, y-axis with target tokens.
    Add colorbar.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# The key sentence: adjective reordering
key_sentence = 'the black cat sits on the mat'
translation, attention = translate(
    key_sentence, encoder, attn_layer, dec_emb, dec_lstm, dec_dense,
    src_w2i, tgt_i2w, src_max_len, tgt_max_len,
)

print(f'EN: {key_sentence}')
print(f'FR: {translation}')
print()

fig = plot_attention_map(key_sentence, translation, attention)
plt.show()

print('Expected:')
print('  - Diagonal bright: each target token attends to its aligned source token')
print('  - Off-diagonal for "noir" (row 2): should attend to "black" (col 2), not "cat" (col 3)')
print('    This is the adjective reordering signature.')


---
## Part 7 (Bonus) — Beam Search

Greedy decoding always picks the single best token at each step.
This can miss sequences that have a lower-probability first token
but a much higher overall probability.

**Beam search** keeps the top `k` partial sequences at each step.


In [ ]:
def beam_search_translate(
    sentence: str,
    encoder: Model,
    attn_layer: 'BahdanauAttention',
    dec_emb: object,
    dec_lstm: object,
    dec_dense: object,
    src_w2i: dict,
    tgt_i2w: dict,
    src_max_len: int,
    tgt_max_len: int,
    beam_width: int = 3,
) -> list[tuple[str, float]]:
    """
    Translate using beam search.

    Parameters
    ----------
    sentence : str
    beam_width : int
        Number of beams to keep at each step.

    Returns
    -------
    list of (translation, log_score) tuples, sorted by score descending.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
beam_results = beam_search_translate(
    'the black cat sits on the mat',
    encoder, attn_layer, dec_emb, dec_lstm, dec_dense,
    src_w2i, tgt_i2w, src_max_len, tgt_max_len,
    beam_width=3,
)

print('Beam search results (k=3):')
for i, (trans, score) in enumerate(beam_results):
    print(f'  {i+1}. score={score:.4f}  {trans}')


---
## Summary


In [ ]:
print('TP-06 — Seq2Seq + Bahdanau Attention')
print(f'Corpus: {len(EN_FR_CORPUS)} EN→FR pairs')
print(f'Vocab: src={SRC_VOCAB}, tgt={TGT_VOCAB}')
print(f'Model: EMB_DIM={EMB_DIM}, LSTM_UNITS={LSTM_UNITS}, EPOCHS={EPOCHS}')
print()
for sent in test_sentences[:3]:
    t, _ = translate(sent, encoder, attn_layer, dec_emb, dec_lstm, dec_dense,
                     src_w2i, tgt_i2w, src_max_len, tgt_max_len)
    print(f'{sent}')
    print(f'  → {t}')
print()
print('Next: Session 07 — Self-attention, Transformers, DistilBERT fine-tuning')
